In [1]:
import pandas as pd
from eutl_scraper import download
from eutl_scraper.extract import extract_projects_from_transactions

In [12]:
df = pd.read_csv("data/normalized/eutl_transactions.csv", parse_dates=["transaction_date"])

C:\Users\abrell\AppData\Local\Temp\ipykernel_3096\879723935.py:1: DtypeWarning: Columns (20,22,23,24,25,26,27,28,29,47,49,50,51,52,53,54,55,56,60,62,65,71,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/normalized/eutl_transactions.csv", parse_dates=["transaction_date"])


In [16]:
df = df[df.transaction_date >= pd.to_datetime("2015-01-01")]

In [17]:
for pref in ["transferring", "acquiring"]:
    for i in range(1, 4):
        print(f"{pref}_account_type{i}:", df[f"{pref}_account_type{i}"].unique())

transferring_account_type1: [121. 100. 120. 110.]
transferring_account_type2: ['121-Person Holding Account' '100-Holding Account' '-'
 '120-Operator Holding Account']
transferring_account_type3: ['0-None' '9-Aircraft Operator Account' '7-Operator Holding Account'
 '12-Trading Account' '-' '8-Person Holding Account'
 '22-International Credit Account' '1-AAU Deposit Account'
 '13-Auction Delivery Account' '15-Aviation Auction Account'
 '21-Aviation Allocation Account' '23-Credit Exchange Account'
 '5-Union Allowance Deletion Account'
 '6-Aviation Surrender Set-Aside Account' '14-Auction Account'
 '2-National Allowance Holding Account' '20-Allocation Account'
 '3-Central Clearing Account' '28-ETS AAU Deposit Account'
 '27-EU AAU Account' '31-ETS Central Clearing Account for CP2']
acquiring_account_type1: [121 100 300 230 120 250 411 210 130]
acquiring_account_type2: ['121-Person Holding Account' '100-Holding Account'
 '300-Retirement Account' '230-Voluntary Cancellation Account (Type 3)'


In [ ]:
pha1 = [120]
pha2 = ['8-Person Holding Account', '12-Trading Account']

pref = "transferring"
lst_df = []
df_trans = df[df[f"{pref}_account_type1"].isin(pha1)]
df_trans

,transaction_id,transaction_type,transaction_date,transaction_status,transferring_registry_name,transferring_account_type1,transferring_account_type2,transferring_account_type3,transferring_account_open_dt,transferring_account_end_of_validity,...,track,expiry_date,amount,acquiring_registry_id,transferring_registry_id,ets_id,acquiring_account_id,acquiring_installation_id,transferring_account_id,transferring_installation_id
149178,ES1065376,7-38,2016-12-15 12:39:04,Completed,Spain,120.0,120-Operator Holding Account,0-None,2005-02-28 00:00:00,NaN,...,1.0,NaN,1763,ES,ES,euets,ES_717,NaN,ES_717,NaN
149197,GB72428,7-38,2018-05-14 13:07:26,Completed,United Kingdom,120.0,120-Operator Holding Account,0-None,2005-08-26 00:00:00,2020-06-08 11:39:41,...,2.0,NaN,17527,GB,GB,euets,GB_345,NaN,GB_345,NaN
149385,IT36499,7-38,2016-12-14 16:28:02,Completed,Italy,120.0,120-Operator Holding Account,0-None,2006-04-04 00:00:00,NaN,...,1.0,NaN,2026,IT,IT,euets,IT_1649,NaN,IT_1649,NaN
149528,RO30125,7-38,2017-12-19 12:42:46,Completed,Romania,120.0,120-Operator Holding Account,0-None,2008-04-18 00:00:00,2020-11-23 08:43:32,...,1.0,NaN,770,RO,RO,euets,RO_242,NaN,RO_242,NaN
149575,NO28845,3-0,2015-01-02 15:11:25,Completed,Norway,120.0,120-Operator Holding Account,0-None,2009-04-02 00:00:00,2017-09-05 14:27:30,...,1.0,NaN,7924,NO,NO,euets,NO_5012248,NO_38,NO_72,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2133191,SE31994,4-0,2015-05-19 13:41:25,Completed,Sweden,120.0,120-Operator Holding Account,0-None,2005-03-03 00:00:00,2017-03-20 15:28:47,...,NaN,NaN,15000,SE,SE,euets,SE_4,NaN,SE_38,NaN
2135032,RO30114,7-38,2017-12-19 11:59:40,Completed,Romania,120.0,120-Operator Holding Account,0-None,2010-01-22 00:00:00,NaN,...,NaN,NaN,2000,RO,RO,euets,RO_384,NaN,RO_384,NaN
2135604,GB72791,4-0,2020-06-08 11:01:38,Completed,United Kingdom,120.0,120-Operator Holding Account,0-None,2005-08-26 00:00:00,2020-06-08 11:39:41,...,2.0,NaN,17527,GB,GB,euets,GB_5018326,NaN,GB_345,NaN
2141093,IT36505,7-38,2016-12-14 16:42:32,Completed,Italy,120.0,120-Operator Holding Account,0-None,2006-04-04 00:00:00,NaN,...,NaN,NaN,136,IT,IT,euets,IT_1720,NaN,IT_1720,NaN


In [7]:
df.transferring_account_type3.unique()

array(['0-None', '9-Aircraft Operator Account',
       '7-Operator Holding Account', '8-Person Holding Account',
       '12-Trading Account', '-', '22-International Credit Account',
       '1-AAU Deposit Account', '4-Gateway Deposit Account',
       '2-National Allowance Holding Account',
       '13-Auction Delivery Account', '15-Aviation Auction Account',
       '21-Aviation Allocation Account', '23-Credit Exchange Account',
       '5-Union Allowance Deletion Account',
       '6-Aviation Surrender Set-Aside Account', '14-Auction Account',
       '20-Allocation Account', '3-Central Clearing Account',
       '28-ETS AAU Deposit Account', '27-EU AAU Account',
       '31-ETS Central Clearing Account for CP2'], dtype=object)

In [7]:
transaction_columns = [
    "transaction_id",
    "transaction_type",
    "transaction_date",
    # "transaction_status",  # drop status as we anyways only observe 'completed'
    "ets_id",
    "originating_registry",
    "acquiring_registry_id",
    "acquiring_account_id",
    "acquiring_installation_id",
    "transferring_registry_id",
    "transferring_account_id",
    "transferring_installation_id",
    "unit_type_description",
    "supp_unit_type_description",
    "amount",
]
df_trans = df[transaction_columns].copy()

KeyError: "['originating_registry'] not in index"

In [ ]:
df_trans["unit_type"] = list(
    zip(df_trans.unit_type_description, df_trans.supp_unit_type_description)
)
df_trans.unit_type.unique()

array([('ERU - Emission Reduction Unit', 'No supplementary unit type'),
       ('ERU - Converted from an RMU', 'No supplementary unit type'),
       ('CER - Certified Emission Reduction Unit converted from an AAU', 'No supplementary unit type'),
       ('RMU - Removal Unit', 'No supplementary unit type'),
       ('AAU - Assigned Amount Unit', 'No supplementary unit type'),
       ('Non-Kyoto Unit', 'EU Aviation Allowances (EUAA)'),
       ('AAU - Assigned Amount Unit', 'Allowance issued for the 2008-2012 period and subsequent 5-year periods and is converted from an AAU'),
       ('Non-Kyoto Unit', 'Swiss Aviation Allowances (CHUA)'),
       ('Non-Kyoto Unit', 'Allowance issued for the 2008 to 2012 and subsequent five-year periods by a Member State that does not have AAUs'),
       ('ERU - Emission Reduction Unit', nan),
       ('tCER - Temporary CER', nan),
       ('Non-Kyoto Unit', 'EU General Allowances (EUA)'),
       ('ERU - Converted from an RMU', nan),
       ('CER - Certified Em

In [ ]:
df_trans.tail()

,transaction_id,transaction_type,transaction_date,ets_id,originating_registry,acquiring_registry_id,acquiring_account_id,acquiring_installation_id,transferring_registry_id,transferring_account_id,transferring_installation_id,unit_type_description,supp_unit_type_description,amount
2142470,EU542433,10-2,2020-01-09 18:14:20,euets,EU,EU,EU_5016380,NaN,ES,ES_5008918,ES_160,Non-Kyoto Unit,EU General Allowances (EUA),879
2142471,FR93298,10-0,2009-05-07 17:26:34,euets,PT,NaN,NaN,NaN,NaN,NaN,NaN,AAU - Assigned Amount Unit,Allowance issued for the 2008-2012 period and ...,2444
2142472,FR10339,10-0,2006-06-01 16:34:39,euets,HU,NaN,NaN,NaN,NaN,NaN,NaN,Non-Kyoto Unit,Allowance issued for the 2005-2007 period and ...,1130
2142473,EU403486,10-2,2017-04-27 13:28:12,euets,EU,EU,EU_5016380,NaN,RO,RO_5010063,RO_110,Non-Kyoto Unit,EU General Allowances (EUA),60109
2142474,EU528943,10-0,2019-08-23 13:43:04,euets,EU,HU,HU_5011776,HU_146,NL,NL_5017659,NaN,Non-Kyoto Unit,EU General Allowances (EUA),1602
